In [30]:
import numpy as np
import pandas as pd


In [31]:
# Load in movies dataset from parent directory

movies = pd.read_csv('../ml-32m/movies.csv')
ratings = pd.read_csv('../ml-32m/ratings.csv')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [32]:
# Use this if files are uploaded on google drive:

'''
from google.colab import drive
drive.mount('/content/drive')

movies = pd.read_csv('../ml-32m/movies.csv')
ratings = pd.read_csv('../ml-32m/ratings.csv')
'''

"\nfrom google.colab import drive\ndrive.mount('/content/drive')\n\nmovies = pd.read_csv('../ml-32m/movies.csv')\nratings = pd.read_csv('../ml-32m/ratings.csv')\n"

In [33]:
# Ratings dataset is too big! Reduce to 200,000 rows (~1250 users)

ratings = ratings.iloc[0:200000]

In [34]:
# Merge datasets on movie ID

df = ratings.merge(movies, on='movieId')


# Create user-movie matrix

user_movie_matrix = df.pivot_table(
    index="userId",
    columns="title",
    values="rating"
)

In [35]:
# Simple recommender function

def recommend_movies(movie_title, top_movies=10):

    # Get ratings for selected movie
    movie_ratings = user_movie_matrix[movie_title]

    # Find correlation between this movie and other
    similar_movies = user_movie_matrix.corrwith(movie_ratings)

    # Convert correlations to dataframe
    corr_df = pd.DataFrame(similar_movies, columns=["correlation"])

    # Sort correlations highest to lowest
    recommendations = corr_df.sort_values(by='correlation', ascending=False)

    # Remove the movie itself
    recommendations = recommendations.drop(movie_title, errors="ignore")

    # Return top movies
    return recommendations.head(top_movies)


In [36]:
recommend_movies('Love Actually (2003)')

/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,correlation
title,
"History Boys, The (2006)",1.0
"Hoax, The (2007)",1.0
102 Dalmatians (2000),1.0
Black Widow (1987),1.0
Starter for 10 (2006),1.0
Stealing Home (1988),1.0
"Yes Men Fix the World, The (2009)",1.0
Young & Beautiful (2013),1.0
Brainscan (1994),1.0


In [37]:
# Slightly more complicated recommender function

def recommend_movies(movie_title, top_movies=10):

    # Get ratings for selected movie
    movie_ratings = user_movie_matrix[movie_title]

    # Find correlation between this movie and other
    similar_movies = user_movie_matrix.corrwith(movie_ratings)

    # Convert correlations to dataframe
    corr_df = pd.DataFrame(similar_movies, columns=["correlation"])

    # Remove NaN values
    corr_df = corr_df.dropna()

    # Count number of ratings per movie
    rating_counts = df.groupby("title")["rating"].count()

    # Add rating counts
    corr_df["num_ratings"] = rating_counts

    # Filter out unpopular movies
    recommendations = corr_df[corr_df["num_ratings"] >= 30].sort_values(
        by="correlation",
        ascending=False)

    # Remove the movie itself
    recommendations = recommendations.drop(movie_title, errors="ignore")

    # Return top movies
    return recommendations.head(top_movies)


In [38]:
recommend_movies('Love Actually (2003)')

/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,correlation,num_ratings
title,,
Murder in the First (1995),1.000000,35
Forget Paris (1995),1.000000,37
"20,000 Leagues Under the Sea (1954)",0.965581,31
Jumper (2008),0.889054,35
Stigmata (1999),0.879453,38
Snake Eyes (1998),0.871570,33
Risky Business (1983),0.871421,45
"To Wong Foo, Thanks for Everything! Julie Newmar (1995)",0.866025,34
End of Days (1999),0.866025,34
